In [1]:
import pandas as pd
from pathlib import Path
import numpy as np

DF_PATH = Path("../data/raw/AmesHousing.csv")
df = pd.read_csv(DF_PATH)

In [2]:
# look at the dataframe
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,0,NaN,NaN,NaN,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,0,NaN,MnPrv,NaN,0,3,2010,WD,Normal,189900


In [3]:
# info about the df
df.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   str    
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   str    
 7   Alley            198 non-null    str    
 8   Lot Shape        2930 non-null   str    
 9   Land Contour     2930 non-null   str    
 10  Utilities        2930 non-null   str    
 11  Lot Config       2930 non-null   str    
 12  Land Slope       2930 non-null   str    
 13  Neighborhood     2930 non-null   str    
 14  Condition 1      2930 non-null   str    
 15  Condition 2      2930 non-null   str    
 16  Bldg Type        2930 non-null   str    
 17  House Style      2930 non

In [4]:
# shape
df.shape

(2930, 82)

In [5]:
# size
df.size

240260

In [6]:
# check duplicates
df.duplicated().sum()

np.int64(0)

In [7]:
# NaN values
df.isna().sum()

Order               0
PID                 0
MS SubClass         0
MS Zoning           0
Lot Frontage      490
                 ... 
Mo Sold             0
Yr Sold             0
Sale Type           0
Sale Condition      0
SalePrice           0
Length: 82, dtype: int64

In [8]:
# filling NaN with None
none_cols = ['Pool QC','Fence','Misc Feature','Alley',
             'Garage Type','Garage Finish','Garage Qual','Garage Cond',
             'Bsmt Qual','Bsmt Cond','Bsmt Exposure','BsmtFin Type 1','BsmtFin Type 2',
             'Fireplace Qu']

for none_col in none_cols:
    df[none_col] = df[none_col].replace(np.nan, "None")

In [9]:
# handling LotFrontage by neighborhood median
df['Lot Frontage'] = df.groupby('Neighborhood')['Lot Frontage'].transform(lambda x: x.fillna(x.median()))

In [10]:
# filling numerical columns with median
num_cols = df.select_dtypes(include="number").columns

for num_col in num_cols:
    df[num_col] = df[num_col].fillna(df[num_col].median())

In [11]:
# checking if all numerical columns have no missing values
df[num_cols].isna().sum()

Order              0
PID                0
MS SubClass        0
Lot Frontage       0
Lot Area           0
Overall Qual       0
Overall Cond       0
Year Built         0
Year Remod/Add     0
Mas Vnr Area       0
BsmtFin SF 1       0
BsmtFin SF 2       0
Bsmt Unf SF        0
Total Bsmt SF      0
1st Flr SF         0
2nd Flr SF         0
Low Qual Fin SF    0
Gr Liv Area        0
Bsmt Full Bath     0
Bsmt Half Bath     0
Full Bath          0
Half Bath          0
Bedroom AbvGr      0
Kitchen AbvGr      0
TotRms AbvGrd      0
Fireplaces         0
Garage Yr Blt      0
Garage Cars        0
Garage Area        0
Wood Deck SF       0
Open Porch SF      0
Enclosed Porch     0
3Ssn Porch         0
Screen Porch       0
Pool Area          0
Misc Val           0
Mo Sold            0
Yr Sold            0
SalePrice          0
dtype: int64

In [12]:
# use mode for NaN values
df.Electrical = df.Electrical.fillna(df.Electrical.mode()[0])

In [13]:
# change data type of object to category
obj_cols = df.select_dtypes(include=["object", "str"]).columns

for obj_col in obj_cols:
    df[obj_col] = df[obj_col].astype("category")

In [14]:
# checking Gr Liv Area for extreme values
indices_to_drop = df[df['Gr Liv Area'] > 4000].index
df = df.drop(index=indices_to_drop)

In [15]:
# other columns to check
cols_to_check = ["Mas Vnr Type", "Mas Vnr Area", "Garage Yr Blt", "Utilities", 
                 "Functional", "MS Zoning", "Kitchen Qual", "Sale Type", "Exterior 1st", 
                 "Exterior 2nd"]

for col_to_check in cols_to_check:
    print(f"{col_to_check} NaN sum: {df[col_to_check].isna().sum()}")

Mas Vnr Type NaN sum: 1774
Mas Vnr Area NaN sum: 0
Garage Yr Blt NaN sum: 0
Utilities NaN sum: 0
Functional NaN sum: 0
MS Zoning NaN sum: 0
Kitchen Qual NaN sum: 0
Sale Type NaN sum: 0
Exterior 1st NaN sum: 0
Exterior 2nd NaN sum: 0


In [16]:
# fill Mas Vnr Type NaN with mode
df["Mas Vnr Type"] = df["Mas Vnr Type"].fillna(df["Mas Vnr Type"].mode()[0])

In [17]:
# check if there are no more NaN values
df.isna().sum().sort_values(ascending=False)

Order             0
PID               0
MS SubClass       0
MS Zoning         0
Lot Frontage      0
                 ..
Mo Sold           0
Yr Sold           0
Sale Type         0
Sale Condition    0
SalePrice         0
Length: 82, dtype: int64

In [18]:
# check final dataframe
df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,None,IR1,Lvl,...,0,None,None,None,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,None,Reg,Lvl,...,0,None,MnPrv,None,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,None,IR1,Lvl,...,0,None,None,Gar2,12500,6,2010,WD,Normal,172000
3,4,526353030,20,RL,93.0,11160,Pave,None,Reg,Lvl,...,0,None,None,None,0,4,2010,WD,Normal,244000
4,5,527105010,60,RL,74.0,13830,Pave,None,IR1,Lvl,...,0,None,MnPrv,None,0,3,2010,WD,Normal,189900


In [19]:
# save the cleaned df
df.to_parquet("../data/processed/ames_housing_cleaned.parquet", index=False)